# 实体记忆（Entity Memory）

> **从每一轮对话中抽取命名实体，维护一个关于人物、地点和偏好的结构化知识库，并把相关实体事实注入每次提示词。这让智能体对用户的世界有了持久、结构化的理解。**

想象一个私人助理，为你提到的每个人、每个地点、每个项目都保留一张联系人卡片。每当你分享新信息（「Sarah 升职了」），他就更新对应的卡片。下次你问起 Sarah，他会抽出这张卡片，给你一个完整的回答。实体记忆就是这样工作的。

前面的记忆技术要么逐字存储对话（缓冲），要么按新近度裁剪（滑动窗口），要么压缩成散文（摘要），要么按语义相似度检索（向量库）。它们都不在结构化层面*理解*对话内容。

**实体记忆**换了一条路。每一轮之后，系统抽取实体（人物、地点、偏好、项目），并为每个实体存储结构化事实。当用户说「我的经理 Sarah 批准了柏林之行」，系统会为 *Sarah*（角色：经理）和 *Berlin*（背景：即将成行的旅行）创建或更新记录。在后面的轮次中，用户再提到 Sarah 或 Berlin 时，相关事实会被检索出来并注入提示词。

**收益是：**智能体不只是*记得*说过什么，它*掌握*用户生活中各个实体的结构化事实。它能回答「你知道关于 Sarah 的哪些信息？」这类问题，而不用翻原始历史记录。

**读完本 notebook，你将理解：**
- 如何用 Claude 的工具使用（函数调用）API 构建实体抽取器。
- 一个按实体积累事实的键值实体存储。
- 如何把实体上下文注入提示词，让回答更有依据。
- 跨会话持久化：保存并重新加载实体知识。
- 实体记忆与滑动窗口在结构化召回任务上的对比。


## 核心概念

- **实体抽取（entity extraction）：**用 LLM（通过工具使用 / 函数调用）识别每轮消息中提到的实体：人物、地点、组织、偏好、项目。这比基于规则的 NER（Named Entity Recognition，命名实体识别，即发现并分类专有名词的任务）更灵活。
- **实体存储（entity store）：**一种键值数据结构，把实体名映射到结构化记录（类型、事实列表、元数据）。把它看作智能体关于用户世界的「通讯录」。
- **事实积累（fact accumulation）：**每当某个实体带着新信息出现，事实列表就增长。你可以检测并处理重复和矛盾。
- **实体上下文注入（entity-context injection）：**每次调用 LLM 前，先查找当前消息中提到的实体，把它们的事实作为结构化上下文放进系统提示词。
- **跨会话持久化（cross-conversation persistence）：**实体存储可以序列化成 JSON，并在新会话中重新加载。这让智能体拥有跨越会话的记忆。
- **消歧（disambiguation）：**处理同名不同实体（比如两个叫「Alex」的人）或同实体不同名（比如「妈妈」和「Linda」）的情况。


## Architecture

<p align="center">
  <img src="../../images/diagrams/07_entity_memory.svg" alt="架构图" width="720"/>
</p>

<details><summary>Mermaid 源码</summary>

```mermaid
flowchart LR
    subgraph Turn["Each Conversation Turn"]
        A["User message"] --> B["Entity Extractor\n(Claude tool-use)"]
        B --> C["Extracted entities\n+ new facts"]
        C --> D["Update\nEntity Store"]
    end

    subgraph Store["Entity Store (key-value)"]
        D --> E[("{ name -> {\n  type, facts[],\n  first_seen,\n  last_seen\n} }")]
    end

    subgraph Response["Response Generation"]
        A --> F["Lookup mentioned\nentities"]
        E --> F
        F --> G["Build prompt:\nsystem + entity context\n+ recent messages"]
        G --> H["LLM\n(Claude)"]
        H --> I["Response"]
    end

    subgraph Persist["Persistence"]
        E --> J["Save to JSON"]
        J --> K["Load in\nnew session"]
        K --> E
    end

    style E fill:#4f46e5,color:#fff
    style H fill:#059669,color:#fff
    style B fill:#d97706,color:#fff
```

</details>

In [ ]:
# 安装所需的包（只需运行一次）
%pip install -q python-dotenv matplotlib numpy deepagents langchain-openai

加载环境变量并初始化 Anthropic 客户端。你需要在 `.env` 文件里配置 `LLM_API_KEY`。


In [ ]:
import os
import json
from datetime import datetime

from dotenv import load_dotenv

load_dotenv()  # 从 .env 读取 LLM_API_KEY / LLM_BASE_URL / LLM_MODEL_NAME

from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent

assert os.getenv("LLM_API_KEY"), "Set LLM_API_KEY in your .env file"

# 任何 OpenAI 兼容端点都可以：用官方 OpenAI 时省略 base_url 即可
llm = ChatOpenAI(
    model=os.getenv("LLM_MODEL_NAME", "gpt-4o-mini"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL"),
)



def to_text(content) -> str:
    """把字符串或内容块列表统一转成纯文本。"""
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "") for block in content if isinstance(block, dict)
    )


print("✓ Model ready")


## 核心实现

我们的实体记忆系统有三个组件：

1. **`EntityStore`：**基于字典的存储，把实体名映射到结构化记录（类型、事实、时间戳）。支持保存 / 加载 JSON，实现跨会话持久化。
2. **`EntityExtractor`：**用 Claude 的工具使用 API 从每条消息中抽取实体和事实。LLM 通过函数调用扮演一个结构化 NER 系统。
3. **`EntityMemory`：**主类，负责编排抽取、存储、上下文注入和聊天。


In [ ]:
class EntityStore:
    """把实体名映射到结构化记录的键值存储。"""

    def __init__(self):
        self.entities: dict[str, dict] = {}

    def update_entity(self, name: str, entity_type: str, facts: list[str]) -> None:
        """用新事实创建或更新一个实体。"""
        key = name.lower().strip()
        now = datetime.now().isoformat()

        if key not in self.entities:
            self.entities[key] = {
                "name": name,
                "type": entity_type,
                "facts": [],
                "first_seen": now,
                "last_seen": now,
            }

        record = self.entities[key]
        record["last_seen"] = now
        # 只添加真正的新事实（用小写比较做简单去重）
        existing_lower = {f.lower() for f in record["facts"]}
        for fact in facts:
            if fact.lower() not in existing_lower:
                record["facts"].append(fact)
                existing_lower.add(fact.lower())

    def get_entity(self, name: str) -> dict | None:
        """按名称检索实体记录。"""
        return self.entities.get(name.lower().strip())

    def find_mentioned(self, text: str) -> list[dict]:
        """找出文本字符串中提到的所有已存储实体。"""
        text_lower = text.lower()
        found = []
        for key, record in self.entities.items():
            if key in text_lower:
                found.append(record)
        return found

    def summary(self) -> str:
        """返回所有已存储实体的格式化摘要。"""
        if not self.entities:
            return "No entities stored."
        lines = []
        for key, record in self.entities.items():
            facts_str = "; ".join(record["facts"])
            lines.append(f"  {record['name']} ({record['type']}): {facts_str}")
        return "\n".join(lines)



我们为 `EntityStore` 添加持久化方法。`save` 方法把所有实体写入一个 JSON 文件，`load` 类方法再把它们读回来。这样智能体的知识就能在会话之间延续。


In [ ]:
# 持久化与检查
def _save(self, path: str) -> None:
    """把实体存储持久化到 JSON 文件。"""
    with open(path, "w") as f:
        json.dump(self.entities, f, indent=2)


@classmethod
def _load(cls, path: str) -> "EntityStore":
    """从 JSON 文件加载实体存储。"""
    store = cls()
    with open(path) as f:
        store.entities = json.load(f)
    return store


def _len(self) -> int:
    return len(self.entities)


def _repr(self) -> str:
    return f"EntityStore({len(self.entities)} entities)"


EntityStore.save = _save
EntityStore.load = _load
EntityStore.__len__ = _len
EntityStore.__repr__ = _repr

print("✓ EntityStore helper methods attached")


我们定义工具 schema（一份 JSON 描述），告诉 Claude 如何调用我们的实体抽取函数。工具使用（tool-use，也叫函数调用）让 LLM 输出结构化数据，而不是自由文本。schema 规定了要抽取的字段：实体名、类型和事实列表。


In [ ]:
# 定义实体抽取的工具 schema（OpenAI 工具调用格式）
EXTRACT_ENTITIES_TOOL = {
    "type": "function",
    "function": {
        "name": "store_entities",
        "description": (
            "Extract and store named entities mentioned in the conversation. "
            "Call this tool with ALL entities (people, places, organizations, "
            "preferences, projects, pets, etc.) and their associated facts."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "entities": {
                    "type": "array",
                    "description": "List of entities extracted from the message.",
                    "items": {
                        "type": "object",
                        "properties": {
                            "name": {
                                "type": "string",
                                "description": "The entity name (e.g., 'Sarah', 'Berlin', 'Project Atlas').",
                            },
                            "entity_type": {
                                "type": "string",
                                "enum": ["person", "place", "organization", "preference",
                                         "project", "pet", "event", "other"],
                                "description": "The type/category of this entity.",
                            },
                            "facts": {
                                "type": "array",
                                "items": {"type": "string"},
                                "description": "List of facts learned about this entity from the message.",
                            },
                        },
                        "required": ["name", "entity_type", "facts"],
                    },
                },
            },
            "required": ["entities"],
        },
    },
}


`EntityExtractor` 封装了一次 Claude API 调用，使用上面的工具 schema。它把这段对话发给 Claude，要求抽取所有实体。Claude 通过工具使用接口返回结构化 JSON。


In [ ]:
class EntityExtractor:
    """用工具调用从消息中抽取实体和事实。"""

    def __init__(self, model):
        # tool_choice="required" 强制模型必须调用工具
        self.model = model.bind_tools([EXTRACT_ENTITIES_TOOL], tool_choice="required")

    def extract(self, user_msg: str, assistant_msg: str) -> list[dict]:
        """从一段用户-助手对话中抽取实体。

        返回一个字典列表，键为：name、entity_type、facts。
        """
        response = self.model.invoke([
            {"role": "system", "content": (
                "You are an entity extraction system. Analyze the conversation "
                "and extract ALL named entities (people, places, organizations, "
                "preferences, projects, pets, events, etc.) along with any facts "
                "learned about them. Use concise, consistent entity names "
                "(e.g., 'Alice' not 'my friend Alice'). "
                "If no entities are found, call store_entities with an empty list."
            )},
            {"role": "user", "content": (
                f"Extract entities from this exchange:\n\n"
                f"User: {user_msg}\n"
                f"Assistant: {assistant_msg}"
            )},
        ])

        for call in response.tool_calls or []:
            if call["name"] == "store_entities":
                return call["args"].get("entities", [])
        return []


print("✓ EntityExtractor class defined (uses tool calling)")


`EntityMemory` 类负责编排一切。它持有一个 `EntityStore` 和一个 `EntityExtractor`。构建系统提示词时，它会查找用户当前消息中提到的实体，把它们的事实作为上下文注入。


In [ ]:
class EntityMemory:
    """带实体记忆的聊天智能体：抽取、存储并注入实体上下文。"""

    def __init__(
        self,
        model,
        system_prompt: str | None = None,
        recent_buffer_size: int = 10,
        entity_store: EntityStore | None = None,
    ):
        self.model = model
        self.base_system_prompt = system_prompt or "You are a helpful assistant."
        self.recent_buffer_size = recent_buffer_size

        # 组件
        self.store = entity_store or EntityStore()
        self.extractor = EntityExtractor(model=model)

        # 状态
        self.messages: list[dict] = []
        self.turn_count = 0

    def _build_system_prompt(self, user_input: str) -> str:
        """把相关实体上下文注入系统提示词。"""
        parts = [self.base_system_prompt]

        # 查找当前消息中提到的实体
        mentioned = self.store.find_mentioned(user_input)
        if mentioned:
            entity_context = "\n".join(
                f"- {r['name']} ({r['type']}): {'; '.join(r['facts'])}"
                for r in mentioned
            )
            parts.append(
                f"You have the following knowledge about entities mentioned "
                f"by the user:\n\n{entity_context}\n\n"
                f"Use this knowledge to give informed, personalized responses."
            )

        # 同时简要列出所有已知实体
        if self.store.entities:
            all_names = ", ".join(
                r["name"] for r in self.store.entities.values()
            )
            parts.append(f"All known entities: {all_names}")

        return "\n\n".join(parts)


`chat` 方法是主循环。它构建带实体上下文的提示词，调用 Claude 生成回答，然后对这轮对话做实体抽取。任何新实体或新事实都会存下来，供后续轮次使用。


In [ ]:
# ── 对话 ──────────────────────────────────────────────────────
def _chat(self, user_input: str) -> str:
    """发送消息，抽取实体，并结合实体上下文生成回答。"""
    self.turn_count += 1

    # 构建带实体上下文的提示词
    system = self._build_system_prompt(user_input)

    # 把用户消息加入近期缓冲
    self.messages.append({"role": "user", "content": user_input})
    recent = self.messages[-self.recent_buffer_size:]

    # deepagents 的 invoke 无状态：系统提示词带最新实体上下文，每轮重建
    agent = create_deep_agent(model=self.model, system_prompt=system)
    result = agent.invoke({"messages": recent})
    assistant_text = to_text(result["messages"][-1].content)

    # 把助手回复存入缓冲
    self.messages.append({"role": "assistant", "content": assistant_text})

    # 从这轮对话中抽取实体
    try:
        extracted = self.extractor.extract(user_msg=user_input, assistant_msg=assistant_text)
        for entity in extracted:
            self.store.update_entity(
                name=entity["name"],
                entity_type=entity["entity_type"],
                facts=entity["facts"],
            )
    except Exception as e:
        print(f"  [extraction warning: {e}]")

    return assistant_text


EntityMemory.chat = _chat

print("✓ EntityMemory.chat attached")


我们添加几个实用方法：检查实体存储、保存到磁盘、从文件加载。`from_saved` 类方法从一个 JSON 文件创建预加载好的 `EntityMemory`，这就是跨会话恢复的方式。


In [ ]:
# ── 实用方法 ──────────────────────────────────────────────────
def _known_entities(self) -> str:
    """返回所有已知实体的格式化摘要。"""
    return self.store.summary()


def _save_memory(self, path: str) -> None:
    """把实体存储持久化到磁盘。"""
    self.store.save(path)


@classmethod
def _from_saved(cls, path: str, **kwargs) -> "EntityMemory":
    """创建一个预加载了已保存实体的 EntityMemory 实例。"""
    store = EntityStore.load(path)
    return cls(entity_store=store, **kwargs)


def _repr(self) -> str:
    return (
        f"EntityMemory(turns={self.turn_count}, "
        f"entities={len(self.store)})"
    )


EntityMemory.known_entities = _known_entities
EntityMemory.save_memory = _save_memory
EntityMemory.from_saved = _from_saved
EntityMemory.__repr__ = _repr

print("✓ EntityMemory helper methods attached")


## 使用示例：实体抽取实战

我们进行一段多轮对话，其中提到若干人物、地点和偏好。每轮之后，系统抽取实体并积累结构化事实。后面的轮次会受益于这些知识。


In [ ]:
mem = EntityMemory(
    model=llm,
    system_prompt="You are a friendly personal assistant. Reply concisely in 1-2 sentences.",
)

messages = [
    "Hi! My name is Priya and I'm a data scientist at Spotify.",
    "My partner Alex is a chef -- he runs a Thai restaurant called Basil & Lime in Portland.",
    "We have a rescue greyhound named Zephyr who loves long walks.",
    "I'm working on a project called Wavelength -- it's a music recommendation engine.",
    "Alex and I are planning a trip to Tokyo next spring.",
    "What do you know about Alex?",
    "What project am I working on?",
]

for msg in messages:
    print(f"\U0001f464 User:  {msg}")
    reply = mem.chat(msg)
    print(f"\U0001f916 Agent: {reply}")
    print()

print(f"\n\U0001f4ca Entities tracked: {len(mem.store)}")

我们检查系统抽取的每个实体。每个实体会显示类型、积累的事实，以及首次和最近出现的时间。这就是智能体关于用户世界的结构化「通讯录」。


In [ ]:
# 检查实体存储
print("=== Entity Store Contents ===\n")
for key, record in mem.store.entities.items():
    print(f"\U0001f4cc {record['name']} ({record['type']})")
    for fact in record["facts"]:
        print(f"   \u2022 {fact}")
    print(f"   First seen: {record['first_seen'][:19]}")
    print(f"   Last seen:  {record['last_seen'][:19]}")
    print()

## 跨会话持久化

实体记忆的一大优势：很容易序列化。实体存储就是一个字典。保存成 JSON，在新会话里加载，智能体就能从上次停下的地方继续。

这模拟了用户关闭聊天、第二天再开一个新对话时的情形。


In [ ]:
# -- 保存会话 1 的实体存储 --
ENTITY_FILE = "entity_store.json"
mem.save_memory(ENTITY_FILE)
print(f"\u2713 Saved {len(mem.store)} entities to {ENTITY_FILE}")

# -- 模拟新会话（会话 2） --
print("\n--- NEW SESSION ---\n")

mem2 = EntityMemory.from_saved(
    ENTITY_FILE,
    system_prompt="You are a friendly personal assistant. Reply concisely in 1-2 sentences.",
)
print(f"Loaded entities: {len(mem2.store)}")
print(f"Known entities: {[r['name'] for r in mem2.store.entities.values()]}\n")

# 新会话应该知道 Priya、Alex、Zephyr 等实体的信息
session2_questions = [
    "What's the name of my dog?",
    "Where does Alex work?",
    "Tell me about the Wavelength project.",
]

for msg in session2_questions:
    print(f"\U0001f464 User:  {msg}")
    reply = mem2.chat(msg)
    print(f"\U0001f916 Agent: {reply}")
    print()

print("\u2713 Cross-conversation persistence works -- no re-introduction needed!")

## 实验：实体记忆 vs. 滑动窗口

我们做一个受控测试：

1. 在 **30 轮对话**中埋入 **10 条实体事实**（混在闲聊消息里）。
2. 针对这些实体提出 **10 个召回问题**。
3. 对比**实体记忆**（抽取并存储结构化事实）和**滑动窗口**（只保留最近 10 条消息）。

假设是：实体记忆能从任意轮次召回结构化事实。滑动窗口在事实滑出窗口后就会忘记。


In [ ]:
# 在特定轮次埋入的事实
PLANTED_FACTS = {
    1:  ("My name is Jordan Rivera and I'm 34 years old.",     "jordan",    "name/age"),
    3:  ("My best friend Mika teaches physics at MIT.",         "mika",      "friend"),
    5:  ("I have a cat named Pixel -- she's a black tabby.",    "pixel",     "pet"),
    8:  ("My company is called NovaSpark -- we build AR glasses.", "novaspark", "company"),
    11: ("My mom Linda lives in Tucson and she's a retired nurse.", "linda",  "mother"),
    15: ("I'm training for a triathlon in Lake Tahoe this August.", "tahoe",  "event"),
    18: ("My partner Sam is a violinist in the Chicago Symphony.", "sam",     "partner"),
    22: ("I just bought a cabin near Bend, Oregon.",            "bend",      "property"),
    26: ("My therapist Dr. Patel recommended I start journaling.", "patel",  "therapist"),
    29: ("I'm reading 'Project Hail Mary' by Andy Weir right now.", "hail mary", "book"),
}

FILLER_MESSAGES = [
    "What's the difference between HTTP and HTTPS?",
    "Can you explain quantum entanglement simply?",
    "What's a good recipe for banana bread?",
    "How does a heat pump work?",
    "Tell me about the history of jazz.",
    "What's the tallest mountain in South America?",
    "How do noise-canceling headphones work?",
    "What causes auroras?",
    "Explain the Pythagorean theorem.",
    "What's the oldest known civilization?",
    "How does sourdough starter work?",
    "What is the Doppler effect?",
    "Tell me about coral reefs.",
    "How does memory work in the brain?",
    "What's the speed of light?",
    "How do electric cars regenerate braking energy?",
    "What is dark energy?",
    "Tell me about the Silk Road trade routes.",
    "How do birds navigate during migration?",
    "What is the Krebs cycle?",
]



我们把埋入的事实和闲聊消息交错起来，构成 30 轮对话。然后定义召回问题。每个问题针对一条具体的实体事实，并带一个我们会在答案中检查的关键词。


In [ ]:
# 构建 30 轮对话
conversation_30 = []
filler_idx = 0
for turn in range(1, 31):
    if turn in PLANTED_FACTS:
        msg = PLANTED_FACTS[turn][0]
    else:
        msg = FILLER_MESSAGES[filler_idx % len(FILLER_MESSAGES)]
        filler_idx += 1
    conversation_30.append((turn, msg))

RECALL_QUESTIONS = [
    ("What is my full name?",                    "jordan",      1),
    ("Who is Mika?",                             "physics",     3),
    ("What pet do I have?",                      "pixel",       5),
    ("What's my company called?",                "novaspark",   8),
    ("Tell me about my mom.",                    "linda",       11),
    ("What sporting event am I training for?",   "triathlon",   15),
    ("Who is Sam?",                              "violinist",   18),
    ("Do I own any property?",                   "bend",        22),
    ("Who is Dr. Patel?",                        "therapist",   26),
    ("What book am I reading?",                  "hail mary",   29),
]

print(f"Conversation: {len(conversation_30)} turns")
print(f"Facts planted at turns: {sorted(PLANTED_FACTS.keys())}")
print(f"Recall questions: {len(RECALL_QUESTIONS)}")

我们让 30 轮对话全部跑一遍实体记忆。每轮之后，抽取器识别实体并存储事实。观察实体数量随对话推进而增长。


In [ ]:
# -- 让 30 轮对话跑一遍实体记忆 --

ent_mem = EntityMemory(
    model=llm,
    system_prompt="You are a helpful assistant. Reply concisely in 1-2 sentences.",
)

print("Running 30-turn conversation through Entity Memory...")
for turn_num, msg in conversation_30:
    ent_mem.chat(msg)
    if turn_num % 10 == 0:
        print(f"  Turn {turn_num}/30 done ({len(ent_mem.store)} entities tracked)")

print(f"\n\u2713 Done. Entities tracked: {len(ent_mem.store)}")
print(f"\n=== Extracted Entities ===")
print(ent_mem.known_entities())

我们定义一个滑动窗口基线，让同样的 30 轮对话跑一遍。窗口只保留最近 10 条消息，较早的事实会掉出上下文。


In [ ]:
class SlidingWindowBaseline:
    """滑动窗口：只保留最近 K 条消息。"""

    def __init__(self, model, window_size: int = 10, system_prompt: str | None = None):
        self.agent = create_deep_agent(model=model, system_prompt=system_prompt)
        self.window_size = window_size
        self.messages: list[dict] = []

    def chat(self, user_input: str) -> str:
        self.messages.append({"role": "user", "content": user_input})
        window = self.messages[-self.window_size:]

        result = self.agent.invoke({"messages": list(window)})
        assistant_text = to_text(result["messages"][-1].content)
        self.messages.append({"role": "assistant", "content": assistant_text})
        return assistant_text


# -- 让 30 轮对话跑一遍滑动窗口 --

sw_mem = SlidingWindowBaseline(
    model=llm,
    window_size=10,
    system_prompt="You are a helpful assistant. Reply concisely in 1-2 sentences.",
)

print("Running 30-turn conversation through Sliding Window Memory...")
for turn_num, msg in conversation_30:
    sw_mem.chat(msg)
    if turn_num % 10 == 0:
        print(f"  Turn {turn_num}/30 done")

print(f"\n✓ Done. Window keeps last {sw_mem.window_size} messages.")


现在在两个系统上运行召回测试。对每个问题，我们检查答案是否包含预期的关键词。实体记忆应该能从任意轮次召回事实，因为它永久存储了这些事实。滑动窗口会丢失已经滑出窗口的事实。


In [ ]:
# -- 召回测试：实体记忆 --
print("=== Entity Memory - Recall Test ===\n")
ent_results = []

for question, keyword, planted_at in RECALL_QUESTIONS:
    answer = ent_mem.chat(question)
    recalled = keyword.lower() in answer.lower()
    ent_results.append({
        "question": question, "keyword": keyword,
        "planted_at": planted_at, "recalled": recalled,
        "answer": answer,
    })
    status = "\u2713" if recalled else "\u2717"
    print(f"  {status} (turn {planted_at:2d}) {question}")
    print(f"    Answer: {answer[:120]}")
    print()

# -- 召回测试：滑动窗口 --
print("=== Sliding Window - Recall Test ===\n")
sw_results = []

for question, keyword, planted_at in RECALL_QUESTIONS:
    answer = sw_mem.chat(question)
    recalled = keyword.lower() in answer.lower()
    sw_results.append({
        "question": question, "keyword": keyword,
        "planted_at": planted_at, "recalled": recalled,
        "answer": answer,
    })
    status = "\u2713" if recalled else "\u2717"
    print(f"  {status} (turn {planted_at:2d}) {question}")
    print(f"    Answer: {answer[:120]}")
    print()

ent_score = sum(1 for r in ent_results if r["recalled"])
sw_score = sum(1 for r in sw_results if r["recalled"])
print(f"Entity Memory score:       {ent_score}/{len(RECALL_QUESTIONS)}")
print(f"Sliding Window score:      {sw_score}/{len(RECALL_QUESTIONS)}")

我们把对比画出来。图 1 显示总召回得分。图 2 按事实的新旧程度拆解召回：每个系统分别记住了哪些埋入的事实？


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# -- 图 1：总召回得分 --
ent_score = sum(1 for r in ent_results if r["recalled"])
sw_score = sum(1 for r in sw_results if r["recalled"])
labels = ["Entity\nMemory", "Sliding Window\n(K=10)"]
scores = [ent_score, sw_score]
colors = ["#4f46e5", "#ef4444"]

bars = axes[0].bar(labels, scores, color=colors, width=0.5, alpha=0.85)
axes[0].set_ylabel("Facts Recalled")
axes[0].set_ylim(0, len(RECALL_QUESTIONS) + 1)
axes[0].set_title(f"Total Facts Recalled (out of {len(RECALL_QUESTIONS)})")
axes[0].axhline(y=len(RECALL_QUESTIONS), color="gray", linestyle="--", alpha=0.3)
for bar, score in zip(bars, scores):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 str(score), ha="center", fontweight="bold", fontsize=14)

# -- 图 2：按事实新旧程度的召回 --
fact_turns = [r["planted_at"] for r in ent_results]
ent_recalled = [1 if r["recalled"] else 0 for r in ent_results]
sw_recalled = [1 if r["recalled"] else 0 for r in sw_results]

x = np.arange(len(fact_turns))
width = 0.35
axes[1].bar(x - width/2, ent_recalled, width, label="Entity Memory", color="#4f46e5", alpha=0.85)
axes[1].bar(x + width/2, sw_recalled, width, label="Sliding Window", color="#ef4444", alpha=0.85)
axes[1].set_xlabel("Fact Planted at Turn #")
axes[1].set_ylabel("Recalled? (1=Yes, 0=No)")
axes[1].set_title("Recall by Fact Age")
axes[1].set_xticks(x)
axes[1].set_xticklabels([str(t) for t in fact_turns], fontsize=9)
axes[1].set_yticks([0, 1])
axes[1].set_yticklabels(["No", "Yes"])
axes[1].legend()

plt.tight_layout()
plt.savefig("entity_vs_window.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nEntity Memory:   {ent_score}/{len(RECALL_QUESTIONS)} recalled")
print(f"Sliding Window:  {sw_score}/{len(RECALL_QUESTIONS)} recalled")

## 讨论与权衡

### 优点
- **结构化知识。**与原始缓冲或摘要不同，实体记忆存储的是*关于事物的*事实，而不是对话片段。这让你能精确召回属性（姓名、角色、偏好）。
- **跨会话持久化。**实体存储一行代码就能序列化成 JSON。新会话无需重放历史即可加载已有知识。
- **可组合。**实体记忆与其他技术配合良好。可以用滑动窗口提供近期上下文，用实体提供长期结构化知识。
- **可解释。**你可以确切检查智能体对每个实体「知道」什么。这是一个透明、可审计的知识库。

### 缺点
- **抽取成本。**每轮都要一次额外的 LLM 调用来抽取实体（通过工具使用）。这大致让每轮 API 成本翻倍。
- **抽取不完美。**基于 LLM 的抽取可能漏掉实体、编造事实，或从闲聊中抽出无关实体。
- **消歧。**处理同名不同实体，或同一实体的不同称呼（「妈妈」vs.「Linda」），需要本 notebook 未覆盖的额外逻辑。
- **schema 僵化。**我们的实体类型是固定的。现实中的实体未必能整齐归入某个类别。更灵活的做法是用自由文本存储事实。
- **事实陈旧。**事实只增不减，永不过期。出现矛盾（比如先存了「住在 Portland」，之后用户又说「我搬到 Berlin 了」）时，需要显式的冲突处理。

### 何时使用实体记忆

| 场景 | 建议 |
|----------|---------------|
| 跨会话跟踪用户信息的个人助理 | 非常合适，这是核心用例。 |
| 管理联系人和关系的 CRM 类智能体 | 非常合适。 |
| 没有回头用户的短对话 | 大材小用，滑动窗口就够了。 |
| 大流量、成本敏感的部署 | 谨慎：抽取调用会让成本翻倍。 |
| 需要精确回忆对话流程或措辞 | 改用缓冲或向量库。 |
| 与其他记忆类型组合 | 是滑动窗口 + 摘要的好搭档。 |

### 成本模型
每轮：
- **1 次 LLM 调用**生成回答（与其他方案相同）
- **1 次 LLM 调用**做实体抽取（通过工具使用，输出约 50-200 token）
- **实体存储查找**免费（内存中的字典）

抽取调用是主要的新增成本。在生产环境中，可以考虑异步抽取实体，或者每 *N* 轮批量抽取一次。


## 延伸阅读

- [Anthropic 工具使用（函数调用）](https://docs.anthropic.com/en/docs/build-with-claude/tool-use?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：本 notebook 用于实体抽取的 API
- [LangChain ConversationEntityMemory](https://python.langchain.com/docs/modules/memory/types/entity_summary_memory?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：框架级的实体记忆实现
- [Mem0：自我改进的记忆层](https://github.com/mem0ai/mem0)：带实体跟踪的生产级记忆系统
- [Park et al., "Generative Agents: Interactive Simulacra of Human Behavior" (2023)](https://arxiv.org/abs/2304.03442)：模拟智能体中的实体记忆
- [spaCy 命名实体识别](https://spacy.io/usage/linguistic-features?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques#named-entities)：作为 LLM 抽取替代方案的传统 NER
- [Anthropic：构建对话式 AI](https://docs.anthropic.com/en/docs/build-with-claude/conversational-ai?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：多轮对话模式

---

*← 上一篇：[06 - 向量库记忆](../06_vector_store_memory/) · 下一篇：[08 - 知识图谱记忆](../08_knowledge_graph_memory/) →*


In [ ]:
# 清理临时文件
import os
for f in ["entity_store.json", "entity_vs_window.png"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Removed {f}")

## 🧪 动手试一试

三个小挑战，帮你加深理解。每个大约需要 10-30 分钟。

### 挑战 1：实体别名
给 `EntityStore` 增加一个 `aliases` 字段，让 'Bob'、'Robert' 和 'Bob Smith' 都解析到同一条实体记录。更新 `find_mentioned()` 来检查别名。用一段对同一个人使用多个称呼的对话来测试。

### 挑战 2：抽取的查准率与查全率
准备 10 轮对话，以已知实体和事实作为基准真值。对每一轮运行 `EntityExtractor.extract()`，统计真阳性、假阳性和假阴性，计算查准率（precision）和查全率（recall）。找出抽取器最容易漏掉的实体类型。

### 挑战 3：实体关系跟踪
扩展 `EntityStore`，记录实体之间的关系（比如 'Alice works-at Acme Corp'）。把关系存成每个实体的（主体，谓词，宾语）三元组列表。这直接通向 08 知识图谱记忆的做法。


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--07-entity-memory--entity-memory)
